# Best efforts

Owner-run, local-only comparison of cached best efforts. Each effort is the fastest segment of that distance within one activity, not necessarily an overall personal record. Where no sampled marathon effort is available, a lap-based estimate may be shown; estimates use diamond markers and are not official records. Select a distance to compare its pace across dates. Outputs may contain private activity data; never commit or share them.

In [ ]:
import plotly.graph_objects as go
import polars as pl

from stridelog.db.models import Activity, ActivityAnalytics
from stridelog.notebook_data import read_table
from stridelog.presentation import distance_label, format_duration, pace_ticks

In [ ]:
distances = (400.0, 1_000.0, 5_000.0, 10_000.0, 21_097.5, 42_195.0)
activities = {row["id"]: row for row in read_table(Activity, ["id", "name", "local_date"])}
cache = read_table(ActivityAnalytics, ["activity_id", "metrics"])
rows = [
    {
        "activity": activities[row["activity_id"]]["name"],
        "local_date": activities[row["activity_id"]]["local_date"],
        "source": "Samples",
        **effort,
    }
    for row in cache
    if row["activity_id"] in activities
    for effort in row["metrics"].get("best_efforts", [])
    if effort["distance_m"] in distances
]
for row in cache:
    estimate = row["metrics"].get("lap_estimated_marathon")
    if row["activity_id"] in activities and estimate is not None:
        rows.append(
            {
                "activity": activities[row["activity_id"]]["name"],
                "local_date": activities[row["activity_id"]]["local_date"],
                "source": "Lap estimate",
                **estimate,
            }
        )
if not rows:
    raise ValueError("No cached best efforts are available; run stridelog analyze first.")
pl.DataFrame(
    [
        {
            "activity": effort["activity"],
            "local_date": effort["local_date"],
            "distance": distance_label(effort["distance_m"]),
            "elapsed": format_duration(effort["duration_s"]),
            "pace_min_per_km": format_duration(effort["duration_s"] * 1_000 / effort["distance_m"]),
            "source": effort["source"],
        }
        for effort in sorted(rows, key=lambda effort: (effort["distance_m"], effort["duration_s"]))
    ]
)

In [ ]:
dated = [effort for effort in rows if effort["local_date"] is not None]
if not dated:
    raise ValueError("No cached best efforts have a local date to plot.")
available = {effort["distance_m"] for effort in dated}
selected = next(distance for distance in distances if distance in available)
if 1_000 in available:
    selected = 1_000
figure = go.Figure()
buttons = []
for distance in distances:
    matching = sorted(
        (effort for effort in dated if effort["distance_m"] == distance),
        key=lambda effort: effort["local_date"],
    )
    paces = [effort["duration_s"] * 1_000 / distance for effort in matching]
    label = distance_label(distance)
    tick_values, tick_labels = pace_ticks(paces)
    figure.add_trace(
        go.Scatter(
            x=[effort["local_date"] for effort in matching],
            y=paces,
            mode="markers",
            marker={
                "symbol": [
                    "diamond" if effort["source"] == "Lap estimate" else "circle"
                    for effort in matching
                ]
            },
            name=label,
            visible=distance == selected,
            customdata=[
                [
                    effort["activity"],
                    format_duration(effort["duration_s"]),
                    format_duration(pace),
                    effort["source"],
                ]
                for effort, pace in zip(matching, paces, strict=True)
            ],
            hovertemplate="%{x}<br>%{customdata[0]}<br>Elapsed: %{customdata[1]}"
            "<br>Pace: %{customdata[2]} /km<br>Source: %{customdata[3]}<extra></extra>",
        )
    )
    buttons.append(
        {
            "label": label,
            "method": "update",
            "args": [
                {"visible": [candidate == distance for candidate in distances]},
                {
                    "title.text": f"Best segments per activity: {label}"
                    + ("" if matching else " (unavailable)"),
                    "yaxis.tickvals": tick_values,
                    "yaxis.ticktext": tick_labels,
                },
            ],
        }
    )
selected_button = buttons[distances.index(selected)]
figure.update_layout(
    title=selected_button["args"][1]["title.text"],
    xaxis_title="Run date",
    yaxis_title="Pace (min/km)",
    yaxis={
        "autorange": "reversed",
        "tickmode": "array",
        "tickvals": selected_button["args"][1]["yaxis.tickvals"],
        "ticktext": selected_button["args"][1]["yaxis.ticktext"],
    },
    updatemenus=[{"buttons": buttons, "active": distances.index(selected)}],
)
figure